
Load Python libraries

In [64]:
import pandas as pd

import sys
!{sys.executable} -m pip install scikit-learn pandas matplotlib qiskit scipy

Loads cleaned data in QML notebook

In [65]:
df = pd.read_csv("../data/cleaned_wildfire_data.csv")
df.head()

,avg_tmax_c,avg_tmin_c,tot_prcp_mm,month,temp_range,hot_dry,zip3_890,zip3_894,zip3_895,zip3_900,...,zip3_959,zip3_960,zip3_961,zip3_975,zip3_976,risk,Year,zip,risk_score,predicted_risk
0,NaN,NaN,NaN,7.0,NaN,NaN,False,False,False,False,...,False,False,False,False,False,0,2019,95470.0,NaN,NaN
1,NaN,NaN,NaN,12.0,NaN,NaN,False,False,False,False,...,False,False,False,False,False,0,2018,93060.0,NaN,NaN
2,NaN,NaN,NaN,12.0,NaN,NaN,False,False,False,False,...,False,False,False,False,False,0,2018,93066.0,NaN,NaN
3,21.748387,11.432258,35.5,1.0,10.316129,0.61091,False,False,False,True,...,False,False,False,False,False,0,2018,90001.0,NaN,NaN
4,21.748387,11.432258,35.5,1.0,10.316129,0.61091,False,False,False,True,...,False,False,False,False,False,0,2018,90002.0,NaN,NaN


Selects quantum features (only uses 4 due to hardware limitations)

In [66]:
id_cols = ['zip', 'Year']
q_features = ['avg_tmax_c', 'tot_prcp_mm', 'month', 'hot_dry']
target = 'risk'

train_df = df[df['Year'] < 2023].copy()
test_df = df[df['Year'] == 2023].copy()

Xq_train = train_df[q_features]
yq_train = train_df[target].values

Xq_test = test_df[q_features]
yq_test = test_df[target].values

Fills empty NaN values with median of each column (since some quantum algorithms can't handle NaNs)

In [67]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

Xq_train_imputed = imputer.fit_transform(Xq_train)
Xq_test_imputed = imputer.transform(Xq_test)

Preprocessing for angle encoding on bounded angle range [-pi, pi]

In [68]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))

Xq_train_angles = scaler.fit_transform(Xq_train_imputed)
Xq_test_angles = scaler.transform(Xq_test_imputed)

Xq_train_angles[:3], Xq_test_angles[:3]

(array([[ 0.82031762, -3.11075073,  0.28559933, -2.93652035],
        [ 0.82031762, -3.11075073,  3.14159265, -2.93652035],
        [ 0.82031762, -3.11075073,  3.14159265, -2.93652035]]),
 array([[ 0.82031762, -3.11075073, -1.99919533, -2.93652035],
        [ 0.82031762, -3.11075073, -1.42799666, -2.93652035],
        [ 0.82031762, -3.11075073, -1.42799666, -2.93652035]]))

Shows how imbalanced the data is and then modifies for a 3:1 ratio (more managable for QML)

In [69]:
from sklearn.utils import resample

train_pos_idx = np.where(yq_train == 1)[0]
train_neg_idx = np.where(yq_train == 0)[0]

n_pos_sample = 10
n_neg_sample = 30

Transforms dataset into managable subset for training by randomly selecting positive and negative examples and building feature and label arrays

In [70]:
sampled_pos_idx = resample(train_pos_idx, replace=False, n_samples=n_pos_sample, random_state=42)
sampled_neg_idx = resample(train_neg_idx, replace=False, n_samples=n_neg_sample, random_state=42)

train_idx_small = np.concatenate([sampled_pos_idx, sampled_neg_idx])
np.random.shuffle(train_idx_small)

Xq_train_small = Xq_train_angles[train_idx_small]
Xq_train_small = np.asarray(Xq_train_small, dtype=np.float64)
yq_train_small = yq_train[train_idx_small]
yq_train_small = np.asarray(yq_train_small, dtype=np.float64)

print("Small train set shape:", Xq_train_small.shape)
print("Positive rate:", yq_train_small.mean())

Small train set shape: (40, 4)
Positive rate: 0.25


Imports qml features

In [71]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Pauli

Defines the circuit by turning it into an optimizable function with angle encoding and entanglement

In [72]:
n_qubits = 4
n_layers = 2

def build_qiskit_circuit(x, weights):
    qc = QuantumCircuit(n_qubits)
    
    #angle embedding
    for q in range(n_qubits):
        qc.ry(float(x[q]), q)
    
    #variational layers
    for layer in range(n_layers):
        for q in range(n_qubits):
            qc.rx(float(weights[layer, q, 0]), q)
            qc.ry(float(weights[layer, q, 1]), q)
            qc.rz(float(weights[layer, q, 2]), q)
        #simple ring entanglement
        for q in range(n_qubits - 1):
            qc.cx(q, q+1)
        qc.cx(n_qubits - 1, 0)
    
    return qc

def qiskit_expectation_z0(x, weights):
    qc = build_qiskit_circuit(x, weights)
    state = Statevector.from_instruction(qc)
    obs = Pauli('Z' + 'I' * (n_qubits - 1))
    return np.real(state.expectation_value(obs))

In [73]:
def quantum_model_train(x, weights):
    raw = qiskit_expectation_z0(x, weights)
    prob = (raw + 1) / 2
    return np.clip(prob, 1e-8, 1 - 1e-8)

def quantum_model_infer(x, weights):
    return quantum_model_train(x, weights)

Computes weighted loss so that missing a real wildfire is more costly

In [74]:
eps = 1e-8

pos_weight = (len(yq_train_small) - yq_train_small.sum()) / (yq_train_small.sum() + eps)
pos_weight = float(pos_weight)

Instantiates loss function

In [75]:
def weighted_bce_loss(flat_weights, X, y, pos_weight = 1.0):
    weights = flat_weights.reshape(n_layers, n_qubits, 3)
    losses = []

    for xi, yi in zip(X, y):
        p = quantum_model_train(xi, weights)
        if yi == 1:
            loss = -pos_weight * np.log(p+eps)
        else:
            loss = -np.log(1 - p + eps)
        losses.append(loss)
    return float(np.mean(losses))

Initializes weights

In [76]:
from scipy.optimize import minimize

np.random.seed(42)

initial_weights = 0.1 * np.random.randn(n_layers, n_qubits, 3)
initial_flat = initial_weights.flatten()

Training loop

In [77]:
result = minimize(weighted_bce_loss, initial_flat, args=(Xq_train_small, yq_train_small, pos_weight), method='Powell', options={'maxiter': 200})

weights = result.x.reshape(n_layers, n_qubits, 3)
print("final loss: ", result.fun)

final loss:  0.6582964223877058


Prints some results after training (beyond loss)

In [78]:
def predict_probs_train(X, weights):
    return np.array([float(quantum_model_train(x, weights)) for x in X])

def predict_probs_infer(X, weights):
    return np.array([float(quantum_model_infer(x, weights)) for x in X])

# tiny comparison set
X_compare = Xq_test_angles[:5]
y_compare = yq_test[:5]

In [79]:
import time

start = time.time()
infer_probs = predict_probs_infer(X_compare, weights)
end = time.time()

print("Inference time for 5 samples:", end - start)
print("Infer probs:", infer_probs)

infer_preds = (infer_probs >= 0.5).astype(int)
print("Infer preds:", infer_preds)

Inference time for 5 samples: 0.005154848098754883
Infer probs: [0.28163831 0.35248835 0.35248835 0.35248835 0.35248835]
Infer preds: [0 0 0 0 0]


Zip-level risk scores

In [80]:
test_probs_full_local = predict_probs_train(Xq_test_angles, weights)

results_df = test_df[id_cols + q_features + [target]].copy()
results_df['qiskit_risk_score'] = test_probs_full_local
results_df[['zip', 'Year', 'qiskit_risk_score']].head()

zip_scores = (
    results_df.groupby('zip', as_index=False)['qiskit_risk_score']
    .max()
    .sort_values(['qiskit_risk_score', 'zip'], ascending=[False, True])
)

zip_scores.head(10)
print(zip_scores["qiskit_risk_score"].value_counts().sort_index())

qiskit_risk_score
0.269732     3
0.332456     5
0.352488     2
0.419046     8
0.440844     4
0.502009     6
0.518652    12
0.555006    34
0.561210    45
Name: count, dtype: int64


In [81]:
print("min prob:", np.min(test_probs_full_local))
print("max prob:", np.max(test_probs_full_local))
print("num unique probs:", len(np.unique(test_probs_full_local)))
print("unique probs:", np.unique(np.round(test_probs_full_local, 6))[:20])

min prob: 0.2697321657864197
max prob: 0.5612102345142007
num unique probs: 10
unique probs: [0.269732 0.281638 0.332456 0.352488 0.419046 0.440844 0.502009 0.518652
 0.555006 0.56121 ]


In [82]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(yq_test, test_probs_full_local)
print("ROC AUC:", auc)

ROC AUC: 0.6200131233595801
